# 01 — Data Cleaning & Loading into SQLite
Loads six TMDB CSVs, cleans them (types, duplicates, orphans),
and writes them into `data/tmdb.db`.

In [3]:
import os, sqlite3
import pandas as pd
import numpy as np

DATA_DIR = "data"
DB_PATH  = os.path.join(DATA_DIR, "tmdb.db")
os.makedirs(DATA_DIR, exist_ok=True)

def find_file(*names):
    for n in names:
        p = os.path.join(DATA_DIR, n)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"None of {names} found in {DATA_DIR}")

FILES = {
    "movies":       find_file("movies.csv", "TMDB_data-movies.csv"),
    "genres":       find_file("genres.csv"),
    "movie_genres": find_file("movie_genres.csv"),
    "cast":         find_file("cast.csv"),
    "crew":         find_file("crew.csv"),
    "keywords":     find_file("movie_keywords.csv"),
}
FILES

{'movies': 'data\\movies.csv',
 'genres': 'data\\genres.csv',
 'movie_genres': 'data\\movie_genres.csv',
 'cast': 'data\\cast.csv',
 'crew': 'data\\crew.csv',
 'keywords': 'data\\movie_keywords.csv'}

In [4]:
# Load all six CSVs
movies       = pd.read_csv(FILES["movies"])
genres       = pd.read_csv(FILES["genres"])
movie_genres = pd.read_csv(FILES["movie_genres"])
cast         = pd.read_csv(FILES["cast"])
crew         = pd.read_csv(FILES["crew"])
keywords     = pd.read_csv(FILES["keywords"])

In [5]:
# Inspect shapes, dtypes, nulls
def inspect(name, df):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("\ndtypes:\n", df.dtypes)
    print("\nnulls:\n", df.isnull().sum())

for n, d in [("movies", movies), ("genres", genres), ("movie_genres", movie_genres),
             ("cast", cast), ("crew", crew), ("keywords", keywords)]:
    inspect(n, d)


=== movies ===
shape: (2509, 12)

dtypes:
 movie_id               int64
title                    str
original_language        str
release_date             str
runtime                int64
budget               float64
revenue              float64
popularity           float64
vote_average         float64
vote_count             int64
status                   str
overview                 str
dtype: object

nulls:
 movie_id             0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
status               0
overview             0
dtype: int64

=== genres ===
shape: (19, 2)

dtypes:
 genre_id      int64
genre_name      str
dtype: object

nulls:
 genre_id      0
genre_name    0
dtype: int64

=== movie_genres ===
shape: (6888, 2)

dtypes:
 movie_id    int64
genre_id    int64
dtype: object

nulls:
 movie_id    0
genre_id    0
dtype: int6

### Clean `movies`

In [ ]:
#  Clean the primary key: movie_id
#    movie_id is the unique identifier used to join with other tables
movies["movie_id"] = pd.to_numeric(movies["movie_id"], errors="coerce") 
movies = movies.dropna(subset=["movie_id"])                             
movies["movie_id"] = movies["movie_id"].astype(int)                      

# Parse release_date into a real datetime dtype

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

for col in ["runtime", "budget", "revenue", "vote_count"]:  
    movies[col] = pd.to_numeric(movies[col], errors="coerce") 
for col in ["popularity", "vote_average"]:                    
    movies[col] = pd.to_numeric(movies[col], errors="coerce") 

#  Fill missing values in categorical / text columns
movies["title"]             = movies["title"].fillna("Unknown")      
movies["original_language"] = movies["original_language"].fillna("unknown")  
movies["status"]            = movies["status"].fillna("Unknown")     
movies["overview"]          = movies["overview"].fillna("")          

#  Remove duplicates
#    First pass: exact duplicate rows (caused by scraping/concatenation).
#    Second pass: enforce one row per movie_id (keep first occurrence).
movies = movies.drop_duplicates()                                     
movies = movies.drop_duplicates(subset=["movie_id"], keep="first")    


# NOTE: keep budget/revenue = 0 unchanged (0 = unknown per project brief)

print("movies after cleaning:", movies.shape)

movies after cleaning: (2503, 12)


In [7]:
# ### Clean `genres`
genres["genre_id"] = pd.to_numeric(genres["genre_id"], errors="coerce")
genres = genres.dropna(subset=["genre_id"])
genres["genre_id"] = genres["genre_id"].astype(int)
genres["genre_name"] = genres["genre_name"].fillna("Unknown")
genres = genres.drop_duplicates().drop_duplicates(subset=["genre_id"], keep="first")
print("genres after cleaning:", genres.shape)

genres after cleaning: (19, 2)


In [8]:
# ### Clean `movie_genres`
movie_genres["movie_id"] = pd.to_numeric(movie_genres["movie_id"], errors="coerce")
movie_genres["genre_id"] = pd.to_numeric(movie_genres["genre_id"], errors="coerce")
movie_genres = movie_genres.dropna(subset=["movie_id", "genre_id"])
movie_genres["movie_id"] = movie_genres["movie_id"].astype(int)
movie_genres["genre_id"] = movie_genres["genre_id"].astype(int)
movie_genres = movie_genres.drop_duplicates()
movie_genres = movie_genres[movie_genres.movie_id.isin(movies.movie_id)]
movie_genres = movie_genres[movie_genres.genre_id.isin(genres.genre_id)]
print("movie_genres after cleaning:", movie_genres.shape)

movie_genres after cleaning: (6878, 2)


In [9]:
# ### Clean `cast`
cast["movie_id"]   = pd.to_numeric(cast["movie_id"], errors="coerce")
cast["person_id"]  = pd.to_numeric(cast["person_id"], errors="coerce")
cast["cast_order"] = pd.to_numeric(cast["cast_order"], errors="coerce")
cast = cast.dropna(subset=["movie_id", "person_id"])
cast["movie_id"]    = cast["movie_id"].astype(int)
cast["person_id"]   = cast["person_id"].astype(int)
cast["cast_order"]  = cast["cast_order"].fillna(999).astype(int)
cast["actor_name"]     = cast["actor_name"].fillna("Unknown")
cast["character_name"] = cast["character_name"].fillna("")
cast = cast.drop_duplicates()
cast = cast[cast.movie_id.isin(movies.movie_id)]
cast = cast.reset_index(drop=True)
# Only create surrogate PK if the CSV doesn't already have one
if "cast_row_id" not in cast.columns:
    cast.insert(0, "cast_row_id", cast.index + 1)
print("cast after cleaning:", cast.shape)

cast after cleaning: (24902, 6)


In [10]:
# ### Clean `crew`
crew["movie_id"]  = pd.to_numeric(crew["movie_id"], errors="coerce")
crew["person_id"] = pd.to_numeric(crew["person_id"], errors="coerce")
crew = crew.dropna(subset=["movie_id", "person_id"])
crew["movie_id"]  = crew["movie_id"].astype(int)
crew["person_id"] = crew["person_id"].astype(int)
crew["person_name"] = crew["person_name"].fillna("Unknown")
crew["job"]         = crew["job"].fillna("")
crew["department"]  = crew["department"].fillna("")
crew = crew.drop_duplicates()
crew = crew[crew.movie_id.isin(movies.movie_id)]
crew = crew.reset_index(drop=True)
# Only create surrogate PK if the CSV doesn't already have one
if "crew_row_id" not in crew.columns:
    crew.insert(0, "crew_row_id", crew.index + 1)
print("crew after cleaning:", crew.shape)

crew after cleaning: (7289, 6)


In [11]:
# ### Clean `movie_keywords`
keywords["movie_id"]   = pd.to_numeric(keywords["movie_id"], errors="coerce")
keywords["keyword_id"] = pd.to_numeric(keywords["keyword_id"], errors="coerce")
keywords = keywords.dropna(subset=["movie_id", "keyword_id"])
keywords["movie_id"]   = keywords["movie_id"].astype(int)
keywords["keyword_id"] = keywords["keyword_id"].astype(int)
keywords = keywords.drop_duplicates()
keywords = keywords[keywords.movie_id.isin(movies.movie_id)]
print("keywords after cleaning:", keywords.shape)

keywords after cleaning: (41681, 3)


In [12]:
# ### Create SQLite database with proper schema
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

cur.executescript("""
CREATE TABLE movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    original_language TEXT,
    release_date DATE,
    runtime INTEGER,
    budget BIGINT,
    revenue BIGINT,
    popularity REAL,
    vote_average REAL,
    vote_count INTEGER,
    status TEXT,
    overview TEXT
);
CREATE TABLE genres (
    genre_id INTEGER PRIMARY KEY,
    genre_name TEXT
);
CREATE TABLE movie_genres (
    movie_id INTEGER,
    genre_id INTEGER,
    PRIMARY KEY (movie_id, genre_id)
);
CREATE TABLE cast (
    cast_row_id INTEGER PRIMARY KEY,
    movie_id INTEGER,
    person_id INTEGER,
    actor_name TEXT,
    character_name TEXT,
    cast_order INTEGER
);
CREATE TABLE crew (
    crew_row_id INTEGER PRIMARY KEY,
    movie_id INTEGER,
    person_id INTEGER,
    person_name TEXT,
    job TEXT,
    department TEXT
);
CREATE TABLE movie_keywords (
    movie_id INTEGER,
    keyword_id INTEGER
);
""")
conn.commit()

In [13]:
# ### Insert cleaned data
movies_out = movies[["movie_id","title","original_language","release_date","runtime",
                     "budget","revenue","popularity","vote_average","vote_count","status","overview"]]
movies_out.to_sql("movies", conn, if_exists="append", index=False)

genres[["genre_id","genre_name"]].to_sql("genres", conn, if_exists="append", index=False)
movie_genres[["movie_id","genre_id"]].to_sql("movie_genres", conn, if_exists="append", index=False)
cast[["cast_row_id","movie_id","person_id","actor_name","character_name","cast_order"]].to_sql("cast", conn, if_exists="append", index=False)
crew[["crew_row_id","movie_id","person_id","person_name","job","department"]].to_sql("crew", conn, if_exists="append", index=False)
keywords[["movie_id","keyword_id"]].to_sql("movie_keywords", conn, if_exists="append", index=False)

cur.executescript("""
CREATE INDEX IF NOT EXISTS idx_mg_movie   ON movie_genres(movie_id);
CREATE INDEX IF NOT EXISTS idx_mg_genre   ON movie_genres(genre_id);
CREATE INDEX IF NOT EXISTS idx_cast_movie ON cast(movie_id);
CREATE INDEX IF NOT EXISTS idx_crew_movie ON crew(movie_id);
CREATE INDEX IF NOT EXISTS idx_kw_movie   ON movie_keywords(movie_id);
""")
conn.commit()

In [14]:
# ### Sanity test — JOIN across three tables
test = pd.read_sql("""
SELECT m.title, g.genre_name
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g       ON mg.genre_id = g.genre_id
LIMIT 5
""", conn)
test

,title,genre_name
0,Interstellar,Adventure
1,Interstellar,Drama
2,Interstellar,Science Fiction
3,Inception,Action
4,Inception,Science Fiction


In [15]:
conn.close()
print("✅ Database created at", DB_PATH)

✅ Database created at data\tmdb.db
